# CoREMOF-tools: CR/NCR classification and leakage-safe splitting

This executable notebook is the companion to the [dataset-splitting handbook](../README_DATASET_SPLITTING.md). It shows how to:

1. locate and validate an extracted CoRE-MOF release;
2. inspect current public structure metadata;
3. recompute 3-, 4-, and 5-checker consensus labels;
4. filter structures for a modelling study;
5. merge one or more target files through an explicit JSON contract, audited aliases, and current feature tables;
6. rank finite release fields, joined targets, or features with the audited screening example;
7. inspect direct, reference, CrystalNets-combined, and optional coordinate-and-lattice parent evidence;
8. make a deterministic train/validation/test split without crossing known leakage blocks; and
9. save and verify the assignment CSV and reproducibility receipt.

The splitter reads completed release metadata. It does **not** run the five checkers, calculate RACs/MOFids/Zeo++, edit CIFs, or impute unavailable results. The current null-unresolved MOFid projection is explicitly `STAGE_ONLY`: parent relations built from it are non-published candidates and cannot be promoted by the publication command. Splits from current live or staged v26 inputs remain exploratory and are not official CoRE-MOF benchmark assignments.

## Project-defined split identifiers used in this notebook

The following names are **CoREMOF-tools API identifiers**, not community-standard crystallographic terms:

A **release-authorized parent triad** is one `status/group/size` triple declared in `parent_group_methods.json`, stored in `parent_groups.csv`, and validated by the loader. `MATCHED` means the criterion is available and its observed group has at least two members; `UNMATCHED` means available with one member; `NOT_AVAILABLE` supplies no edge and its size-one group only preserves table shape. The size must equal the number of rows carrying the group. Unavailable rows never match and become unique singletons or explicit exclusions.

Here **exact RAC5** means all 264 ordered finite values match after IEEE-754 binary64 parsing, mapping `-0.0` to `+0.0`, and `float.hex()` serialization (`rtol=atol=0`), with no scaling, deletion, imputation, or rounding. **Exact MOFid** means the complete release-authorized current string matches after conversion to text; Unicode-whitespace collapse; trim; case-insensitive rejection of empty, `-`, `nan`, `none`, `null`, `n/a`, `na`, `unknown`, `missing`, `timeout`, `timed out`, `error`, `failed`, `fail`, `fail process`, `failed process`, or `process failed`; Unicode NFKC; and case-folding, in that order. It is not partial/fuzzy matching and does not alter a CIF, atoms, bonds, occupancies, coordinates, chemistry, or unit cell.

- **`priority_main`** controls explanatory parent assignment. On the complete release it reads `rac_status/rac_group/rac_size`, `mofid2_status/mofid2_group/mofid2_size`, and `mofid1_status/mofid1_group/mofid1_size` from `parent_groups/parent_groups.csv`: (1) each available exact RAC5 group anchors a component; (2) exact MOFid-v2 groups are processed; and (3) exact MOFid-v1 groups are processed. A lower group touching no stronger component creates one; one stronger component receives only unresolved rows; two or more stronger components are never merged, produce `PARENT_METHOD_CONFLICT`, and leave lower-only rows unresolved. Missing all three inputs creates a unique singleton unless `missing_parent="exclude"`. Here priority means parent-evidence precedence, not a queue: it does not rank, schedule, or recalculate failed scientific features. This is not row-wise first-nonmissing selection. It excludes Zeo++, CrystalNets topology, source ID, common name, CIF hash, provisional source-ID/MOFid transitive groups, and StructureMatcher evidence.
- **`main_union`** controls split leakage and is not a parent claim. It reads full CIF SHA-256 values from `manifests/cif_manifest.csv` and source/RAC5/MOFid status/group/size fields from the parent table. Before label, source, variant, metal, structure-ID, or target filtering, it makes transitive components from five exact relations: full CIF SHA-256, database-namespaced source siblings, RAC5, MOFid-v2, and MOFid-v1 equality. A missing full hash fails closed; missing optional evidence adds no edge; nulls never match; and all available edges are unioned with no precedence. The source relation applies the text procedure above separately to `(source_database, source_id)` and retains the database namespace. It can keep distinct `priority_main` groups in one block without calling them the same parent. Zeo++, topology, common name, provisional source-ID/MOFid transitive groups, and StructureMatcher do not enter this graph.
- **`leakage_guard="auto"`** is only a selector: it chooses `main_union` for `priority_main` and `parent_only` for an explicitly selected direct/reference parent method. **`parent_only`** uses only that selected explanatory grouping as blocks and adds no cross-method edge. An unresolved `priority_main` conflict is excluded under `parent_only` but may remain safely assigned and diagnosed under `main_union`.

Compactly: `RAC5 anchors → MOFid v2 attachment → MOFid v1 attachment → singleton/exclusion` explains parent groups; the separate five-edge `main_union` graph controls partition leakage.

## 1. Installation

Run one of the following commands in a terminal, or remove the leading `#` from the appropriate line below. The base classification, target-merge, and splitting API uses only the Python standard library. The optional `full` extra retains historical scientific feature dependencies; it is not needed to consume an already-built release. After installation, run `coremof doctor` to see which optional feature runtimes are available.

In [ ]:
# Install an audited wheel supplied by the project:
# %pip install /path/to/coremof_tools-0.4.0.dev0-py3-none-any.whl

# Editable source installation for developers:
# %pip install -e /path/to/CoRE-MOF-Tools

# Optional historical scientific dependencies:
# %pip install 'CoREMOF-tools[full]'

# Exact target-free representative-diversity backend:
# %pip install 'CoREMOF-tools[benchmark]'

## 2. Imports and portable path configuration

For use on another machine, set `COREMOF_RELEASE` (or the release-specific `COREMOF_V2602_RELEASE`) to the extracted directory containing `dataset_info.json`. Optionally set `COREMOF_NOTEBOOK_OUTPUT` to choose where split files are written.

In [ ]:
import csv
import json
import os
import shlex
import sys
from collections import Counter
from pathlib import Path

from CoREMOF import __version__
from CoREMOF.benchmarks import BenchmarkFeasibilityError
from CoREMOF.cli import doctor
from CoREMOF.dataset import CoREMOFDataset
from CoREMOF.labels import CHECKER_COLUMNS, CHECKER_PRESETS
from CoREMOF.parents import ParentResolver
from CoREMOF.targets import (
    AliasRegistry,
    CURRENT_FEATURE_TABLES,
    TargetSource,
    merge_targets_from_config,
)

print("Python:", sys.version.split()[0])
print("CoREMOF-tools version:", __version__)
assert doctor() == 0

In [ ]:
def find_release_root():
    candidates = []
    configured = os.environ.get("COREMOF_RELEASE") or os.environ.get("COREMOF_V2602_RELEASE")
    if configured:
        candidates.append(Path(configured).expanduser())
    candidates.extend([
        Path.cwd() / "coremof_v26.0.2",
        Path.cwd().parent / "coremof_v26.0.2",
    ])
    for candidate in candidates:
        if (candidate / "dataset_info.json").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "No extracted CoRE-MOF release was found. Set COREMOF_RELEASE "
        "to the directory containing dataset_info.json."
    )

RELEASE_ROOT = find_release_root()
OUTPUT_ROOT = Path(
    os.environ.get("COREMOF_NOTEBOOK_OUTPUT", Path.cwd() / "coremof_notebook_outputs")
).expanduser().resolve()
RUN_STRICT_CIF_CHECK = False

print("Release root:", RELEASE_ROOT)
print("Output root: ", OUTPUT_ROOT)

## 3. Load and validate the release

Normal loading validates the release tables, declared manifest entries, IDs, labels, and parent-group contract. It does not read and hash every CIF byte. That more expensive check is available separately below.

In [ ]:
dataset = CoREMOFDataset.from_release(
    RELEASE_ROOT, verify_cif_files=RUN_STRICT_CIF_CHECK
)

print("Dataset version:    ", dataset.dataset_version)
print("Structures:         ", f"{len(dataset):,}")
print("Release status:     ", dataset.dataset_info.get("release_status"))
print("Parent input status:", dataset.parent_group_methods.get("release_status"))
print("CIF bytes verified: ", dataset.cif_files_verified)
print("Input tables hashed: ", len(dataset.input_hashes))

### Optional strict CIF-byte validation

Set `RUN_STRICT_CIF_CHECK = True` in the configuration cell above when you want the primary dataset object to rehash every CIF and compare it with `manifests/cif_manifest.csv`. This is recommended once for a newly copied or extracted release, but is disabled by default in the tutorial because it reads the complete CIF corpus. The verification state is carried into the split receipt.

In [ ]:
if RUN_STRICT_CIF_CHECK:
    assert dataset.cif_files_verified
    print(f"Verified all {len(dataset):,} CIF files in the primary dataset object.")
else:
    print("Skipped strict CIF-byte validation.")

## 4. Inspect one structure

A `StructureRecord` joins the main metadata row, criterion-specific parent memberships, and the CIF manifest entry. Values read from the public metadata CSV are strings. The five checker statuses are votes or explicit non-votes; they are not error-severity scores.

In [ ]:
record = dataset[0]
checker_statuses = {
    checker: record.metadata[column]
    for checker, column in CHECKER_COLUMNS.items()
}
parent_preview = {}
for method in ("rac5_zeo", "rac5", "zeo", "source_id", "mofid_v2", "mofid_v1"):
    group = record.parent_group(method)
    parent_preview[method] = {
        "status": group.status,
        "group_id": group.group_id,
        "size": group.size,
    }

structure_summary = {
    "structure_id": record.structure_id,
    "source_database": record.source_database,
    "source_id": record.get("source_id"),
    "structure_variant": record.structure_variant,
    "metal_elements": record.metal_elements,
    "doi": record.get("doi"),
    "mofid_v1": record.get("mofid_v1"),
    "mofid_v2": record.get("mofid_v2"),
    "checker_statuses": checker_statuses,
    "parent_groups": parent_preview,
    "cif_manifest": dict(record.cif_manifest) if record.cif_manifest else None,
}
print(json.dumps(structure_summary, indent=2))

## 5. Recompute CR/NCR consensus

For every selected checker set, the rule is strict:

- all included checkers `PASS` → `CR`;
- all included checkers `FAIL` → `NCR`;
- a complete mixture of `PASS` and `FAIL` → `AMBIGUOUS`;
- any unavailable/error/timeout non-vote → `UNCHECKED`.

An execution error is never converted into `FAIL` or `NCR`.

In [ ]:
classified_views = {}
label_order = ("CR", "NCR", "AMBIGUOUS", "UNCHECKED")

print("view       checkers  CR       NCR      AMBIGUOUS  UNCHECKED")
for view_name in ("3checker", "4checker", "5checker"):
    view = dataset.classify(view_name)
    classified_views[view_name] = view
    counts = view.label_counts()
    values = [counts.get(label, 0) for label in label_order]
    print(f"{view_name:<11}{len(CHECKER_PRESETS[view_name]):<10}" + "".join(f"{value:<10}" for value in values))
    assert sum(values) == len(dataset)

five_checker = classified_views["5checker"]

### Optional user-defined checker experiment

Python accepts an explicit ordered checker sequence. Such a result is marked `USER_DEFINED` and must not be described as an official release checker view.

In [ ]:
custom_view = dataset.classify(("MOFClassifier", "Chen-Manz"))
print("Identifier:    ", custom_view.checker_view)
print("Official view: ", custom_view.checker_view_official)
print("Label counts:  ", dict(custom_view.label_counts()))
assert not custom_view.checker_view_official

## 6. Filter a modelling cohort

The following example keeps classified CR/NCR structures from the COD or SI sources, uses ASR/FSR variants, and requires Cu or Zn. Different filter categories are combined with AND; Cu/Zn are combined with OR. Filtering does not erase full-release parent bridges used by the recommended leakage guard.

In [ ]:
modelling_subset = five_checker.filter(
    labels=("CR", "NCR"),
    sources=("COD", "SI"),
    variants=("ASR", "FSR"),
    metals=("Cu", "Zn"),
)

source_counts = Counter(record.source_database for record in modelling_subset)
print("Selected structures:", f"{len(modelling_subset):,}")
print("Labels:             ", dict(modelling_subset.label_counts()))
print("Sources:            ", dict(sorted(source_counts.items())))
print("First five IDs:     ", modelling_subset.structure_ids[:5])

## 7. Merge target results before splitting

Target data must be joined **before** splitting so rows without required endpoints are excluded before partition assignment. The leakage graph still uses the complete release universe, including filtered-out and target-missing structures.

Two project-defined CoREMOF-tools identifiers appear in the next code cell; neither is a community-standard crystallographic term:

- **`priority_main`** is the explanatory parent hierarchy. On the complete release it first creates components from every available release-authorized exact RAC5 group. It next processes exact MOFid-v2 groups and finally exact MOFid-v1 groups. A lower-priority group may attach unresolved rows to zero or one stronger component. If it touches two or more stronger components, those components are never merged: the package records `PARENT_METHOD_CONFLICT`, and lower-only rows remain unresolved. A structure with none of these three inputs becomes its own unique singleton unless explicit exclusion is requested. Here priority means parent-evidence precedence, not a queue: it does not rank, schedule, or recalculate failed scientific features. Zeo++, CrystalNets topology, source ID, common name, CIF hash, and StructureMatcher do not enter `priority_main`.
- **`main_union`** is the separate conservative leakage guard, not a parent claim. It forms transitive connected components over the complete unfiltered release using exact full CIF SHA-256, database-namespaced source-sibling, RAC5, MOFid-v2, and MOFid-v1 edges. `leakage_guard="auto"` selects `main_union` for `priority_main`. Thus two different explanatory parent groups may be kept in one indivisible split block without being called the same parent.

In short: `RAC5 anchors → MOFid v2 attachment → MOFid v1 attachment → singleton/exclusion` explains parent groups, while the broader five-edge `main_union` graph controls leakage. Both are built before experiment filters.

The portable example below creates two small target files plus an explicit alias registry. It demonstrates CSV + JSONL input, canonical target names, declared units/conditions/types, exact earlier-ID mapping, current feature-table joins, a reusable JSON target configuration, and hash-bound outputs. The alias is illustrative; real earlier IDs must come from an audited registry. There is no fuzzy name matching or unit/condition inference.

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
target_example_ids = (five_checker.cr_ids + five_checker.ncr_ids)[:8]
assert target_example_ids, "The release has no 5-checker CR/NCR rows for this example."
uptake_path = OUTPUT_ROOT / "example_uptake.csv"
selectivity_path = OUTPUT_ROOT / "example_selectivity.jsonl"
alias_path = OUTPUT_ROOT / "example_alias_registry.csv"
target_config_path = OUTPUT_ROOT / "example_target_config.json"
illustrative_alias = "EARLIER-DEMO-0001"

with alias_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=("structure_id", "earlier_id"))
    writer.writeheader()
    for index, structure_id in enumerate(target_example_ids):
        writer.writerow({
            "structure_id": structure_id,
            "earlier_id": illustrative_alias if index == 0 else "",
        })

with uptake_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=("structure_name", "uptake"))
    writer.writeheader()
    for index, structure_id in enumerate(target_example_ids, start=1):
        input_id = illustrative_alias if index == 1 else structure_id
        writer.writerow({"structure_name": input_id, "uptake": index / 10})

selectivity_ids = target_example_ids[:-2] if len(target_example_ids) > 3 else target_example_ids[:1]
with selectivity_path.open("w", encoding="utf-8") as handle:
    for index, structure_id in enumerate(selectivity_ids, start=1):
        handle.write(json.dumps({"structure_id": structure_id, "selectivity": index * 2.0}) + "\n")

available_feature_tables = tuple(
    name
    for name, relative_path in CURRENT_FEATURE_TABLES.items()
    if (RELEASE_ROOT / relative_path).is_file()
)
target_config = {
    "sources": [
        {
            "path": uptake_path.name,
            "name": "portable_uptake",
            "id_column": "structure_name",
            "target_columns": ["uptake"],
            "target_names": {"uptake": "xe_uptake"},
            "value_types": {"xe_uptake": "float"},
            "units": {"xe_uptake": "mol/kg"},
            "conditions": {"xe_uptake": {"temperature_K": 298, "pressure_bar": 1}},
        },
        {
            "path": selectivity_path.name,
            "name": "portable_selectivity",
            "target_columns": ["selectivity"],
            "target_names": {"selectivity": "xe_kr_selectivity"},
            "value_types": {"xe_kr_selectivity": "float"},
            "units": {"xe_kr_selectivity": "dimensionless"},
            "conditions": {"xe_kr_selectivity": {"temperature_K": 298, "feed": {"Xe": 0.2, "Kr": 0.8}}},
        },
    ],
    "alias_registry": {
        "path": alias_path.name,
        "current_id_column": "structure_id",
        "alias_columns": ["earlier_id"],
    },
    "feature_tables": list(available_feature_tables),
}
target_config_path.write_text(
    json.dumps(target_config, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

# The equivalent object API uses AliasRegistry(...) and TargetSource(...).
alias_registry = AliasRegistry(alias_path, alias_columns=("earlier_id",))
portable_source = TargetSource(
    uptake_path, id_column="structure_name", target_columns=("uptake",),
    target_names={"uptake": "xe_uptake"},
    value_types={"xe_uptake": "float"}, units={"xe_uptake": "mol/kg"},
)
assert alias_registry.alias_columns == ("earlier_id",)
assert portable_source.target_names["uptake"] == "xe_uptake"

target_dataset = merge_targets_from_config(dataset, target_config_path)
target_receipt = target_dataset.receipt()
print("Target columns:      ", target_dataset.target_columns)
print("Feature table names:", available_feature_tables)
print("Feature columns:     ", len(target_dataset.feature_columns))
print("Alias-resolved rows: ", target_receipt["counts"]["alias_id_rows"])
print("First values:        ", dict(target_dataset.target_values(target_example_ids[0])))

In [ ]:
target_split = target_dataset.classify("5checker").train_valid_test_split(
    parent_method="priority_main",
    leakage_guard="auto",
    labels=("CR", "NCR"),
    required_targets=("xe_uptake", "xe_kr_selectivity"),
    required_target_mode="all",
    fractions=(1.0, 0.0, 0.0),
    random_state=42,
)
expected_target_ids = set(selectivity_ids)
assert set(target_split.assignments) == expected_target_ids
assert target_split.leakage_audit["passed"]
assert target_split.receipt()["filters"]["targets"]["filter_precedes_assignment"]
assert target_split.receipt()["filters"]["targets"]["leakage_blocks_use_full_release_universe"]
target_filter_receipt = target_split.receipt()["filters"]["targets"]
print("Rows with both targets:     ", len(target_split.assignments))
print("Rows excluded before split: ", target_filter_receipt["excluded_release_count"])
print("Target digest:              ", target_dataset.receipt()["target_values_sha256"][:16])

### Save the merged modelling table

The merge writer emits a current-ID CSV, observation-level provenance JSONL, and a receipt that binds source/config/alias/feature hashes and scientific target definitions. Keep these together. Nulls remain null; conflicting duplicate values, ambiguous aliases, unknown IDs, or conflicting units/conditions fail the entire merge.

In [ ]:
merged_csv, merged_provenance, merged_receipt = target_dataset.write(
    OUTPUT_ROOT, stem="portable_target_features", overwrite=True
)
print("Merged CSV:       ", merged_csv)
print("Provenance JSONL:", merged_provenance)
print("Merge receipt:    ", merged_receipt)

### Optional real local Rosenbluth-target workflow

Set `COREMOF_ROSENBLUTH_TARGET_CONFIG` to the audited local JSON configuration before running this section. The example is skipped when the variable is unset or the file is absent, so the notebook remains portable. These values are **dimensionless Rosenbluth weights**, not uptake, Henry coefficients, or selectivity. The real configuration must declare the calculation conditions and requested feature tables explicitly.

As dated audit evidence—not a permanent acceptance constant—the portable **2026-08-05** replay covered **42,574 release rows**, found **17,245 rows complete for all configured targets**, bound **137,960 target observations**, and joined **308 feature columns**. The target-first 5-checker COD/SI `priority_main`/`auto` replay assigned **3,828 train / 479 validation / 479 test** structures with **zero crossed leakage blocks**. The printed receipt summary below lets a later run be compared without exposing private absolute paths.

In [ ]:
rosenbluth_config_value = os.environ.get("COREMOF_ROSENBLUTH_TARGET_CONFIG")
rosenbluth_config_path = (
    Path(rosenbluth_config_value).expanduser() if rosenbluth_config_value else None
)
real_rosenbluth_dataset = None
real_rosenbluth_split = None
real_rosenbluth_summary = None

if rosenbluth_config_path is not None and rosenbluth_config_path.is_file():
    real_rosenbluth_dataset = merge_targets_from_config(dataset, rosenbluth_config_path)
    required_rosenbluth = ("co2_rosenbluth_weight", "n2_rosenbluth_weight")
    missing_columns = set(required_rosenbluth).difference(real_rosenbluth_dataset.target_columns)
    if missing_columns:
        raise ValueError("Rosenbluth config lacks required targets: " + ", ".join(sorted(missing_columns)))
    real_rosenbluth_split = real_rosenbluth_dataset.classify("5checker").train_valid_test_split(
        parent_method="priority_main",
        leakage_guard="auto",
        labels=("CR", "NCR"),
        sources=("COD", "SI"),
        required_targets=required_rosenbluth,
        required_target_mode="all",
        random_state=42,
    )
    assert real_rosenbluth_split.receipt()["filters"]["targets"]["filter_precedes_assignment"]
    assert real_rosenbluth_split.leakage_audit["passed"]

    real_merge_receipt = real_rosenbluth_dataset.receipt()
    real_feature_receipts = tuple(real_merge_receipt.get("feature_tables", ()))
    real_alias_receipt = real_merge_receipt.get("alias_registry")
    real_config_receipt = real_merge_receipt.get("config") or {}
    real_rosenbluth_summary = {
        "release_structure_count": real_merge_receipt["release_structure_count"],
        "counts": dict(real_merge_receipt["counts"]),
        "target_columns": list(real_merge_receipt["target_columns"]),
        "feature_table_count": len(real_feature_receipts),
        "feature_column_count": sum(int(item["column_count"]) for item in real_feature_receipts),
        "sources": [
            {"file_name": Path(str(item["file_name"])).name, "sha256": item["sha256"]}
            for item in real_merge_receipt["sources"]
        ],
        "alias_registry_sha256": (real_alias_receipt or {}).get("sha256"),
        "config_sha256": real_config_receipt.get("sha256"),
        "target_values_sha256": real_merge_receipt["target_values_sha256"],
        "split_counts": dict(real_rosenbluth_split.counts),
        "cross_split_block_count": real_rosenbluth_split.leakage_audit["cross_split_block_count"],
    }
    print(json.dumps(real_rosenbluth_summary, indent=2, sort_keys=True))
else:
    print("Skipped real Rosenbluth example; set COREMOF_ROSENBLUTH_TARGET_CONFIG to an existing config file.")

## 8. Rank finite candidates with the frozen screening example

[`screen_candidates.py`](screen_candidates.py) is the standard-library-only command-line companion to this notebook. It validates the release, optionally performs the same configured target/feature merge, recomputes the requested checker consensus, applies checker/source/variant/metal/required-target filters **before** ranking, and writes a ranked CSV plus a hash-bound receipt. The receipt records `policies.filter_precedes_ranking = true`. Missing, non-numeric, and non-finite ranking values are excluded rather than imputed; ties are resolved by `structure_id` ascending. Ranking never changes a scientific label.

Different filter categories are combined with AND. Repeating `--source`, `--variant`, `--metal`, `--label`, or `--require-target` selects multiple values within that category; repeated required endpoints use `--required-target-mode all` by default. `--limit` is applied only after deterministic ranking. The optional `--split` output includes only emitted candidates, while the project-defined `priority_main` explanatory hierarchy with `--leakage-guard auto` still constructs the distinct project-defined `main_union` leakage graph from the complete release universe, exactly as defined in Section 7. Remove `--split` and its split-specific options when only a ranked table is needed.

The cells below only **display** commands for review; they do not launch subprocesses. First, rank a numeric field already present in release metadata:

In [ ]:
screening_script_candidates = (
    Path.cwd() / "examples" / "screen_candidates.py",
    Path.cwd() / "screen_candidates.py",
)
SCREENING_SCRIPT = next(
    (path.resolve() for path in screening_script_candidates if path.is_file()),
    Path("examples/screen_candidates.py"),
)
metadata_screen_command = [
    sys.executable, str(SCREENING_SCRIPT), str(RELEASE_ROOT),
    "--rank-by", "cell_volume_A3",
    "--order", "descending",
    "--checkers", "5checker",
    "--label", "CR",
    "--source", "COD",
    "--output-directory", str(OUTPUT_ROOT / "screening_cell_volume"),
    "--stem", "cod_cr_cell_volume",
]
print(shlex.join(metadata_screen_command))

Next, reuse the generated target configuration to rank `xe_uptake`, require both endpoints, retain 5-checker CR/COD/ASR/Cu rows, cap the result at 100 candidates, and request a `priority_main`/`auto` leakage-safe split:

In [ ]:
target_screen_command = [
    sys.executable, str(SCREENING_SCRIPT), str(RELEASE_ROOT),
    "--target-config", str(target_config_path),
    "--rank-by", "xe_uptake",
    "--require-target", "xe_uptake",
    "--require-target", "xe_kr_selectivity",
    "--required-target-mode", "all",
    "--checkers", "5checker",
    "--label", "CR",
    "--source", "COD",
    "--variant", "ASR",
    "--metal", "Cu",
    "--order", "descending",
    "--limit", "100",
    "--split",
    "--parent-method", "priority_main",
    "--leakage-guard", "auto",
    "--missing-parent", "singleton",
    "--random-state", "42",
    "--output-directory", str(OUTPUT_ROOT / "screening_xe_uptake"),
    "--stem", "cod_asr_cu_xe_uptake",
]
print(shlex.join(target_screen_command))

### Guarded command for the real local Rosenbluth configuration

The following command is constructed only when `COREMOF_ROSENBLUTH_TARGET_CONFIG` names an existing file. **Descending Rosenbluth weight is a workflow demonstration, not a claim about uptake or selectivity performance.** Choose and justify the ranking objective scientifically for an actual screen.

In [ ]:
rosenbluth_screen_command = None
if rosenbluth_config_path is not None and rosenbluth_config_path.is_file():
    rosenbluth_screen_command = [
        sys.executable, str(SCREENING_SCRIPT), str(RELEASE_ROOT),
        "--target-config", str(rosenbluth_config_path),
        "--rank-by", "co2_rosenbluth_weight",
        "--require-target", "co2_rosenbluth_weight",
        "--require-target", "n2_rosenbluth_weight",
        "--required-target-mode", "all",
        "--checkers", "5checker",
        "--label", "CR",
        "--source", "COD",
        "--source", "SI",
        "--order", "descending",
        "--limit", "1000",
        "--split",
        "--parent-method", "priority_main",
        "--leakage-guard", "auto",
        "--output-directory", str(OUTPUT_ROOT / "screening_rosenbluth"),
        "--stem", "cod_si_co2_rosenbluth_demo",
    ]
    print("WARNING: descending Rosenbluth weight is a workflow demonstration, not uptake/selectivity performance.")
    print(shlex.join(rosenbluth_screen_command))
else:
    print("Skipped Rosenbluth screening command; COREMOF_ROSENBLUTH_TARGET_CONFIG is unset or missing.")

## 9. Inspect the recommended parent hierarchy

`priority_main` is the project-defined conflict-aware explanatory hierarchy defined before its first use in Section 7: exact release-authorized RAC5 anchors → exact MOFid v2 attachment → exact MOFid v1 attachment. Here priority means parent-evidence precedence, not a queue: it does not rank, schedule, or recalculate failed scientific features. It is not a simple row-wise first-nonmissing fallback. A lower group never merges multiple stronger components, and its conflict is retained explicitly. Resolution is constructed over the complete release before the requested preview is returned. Missing evidence becomes a unique singleton by default, so two null values never match. It excludes Zeo++, topology, source ID, common name, CIF hash, and StructureMatcher evidence.

Here **canonicalized identifier text** means exactly: convert a release-authorized current value to text; collapse each Unicode-whitespace run to one ASCII space; trim it; reject empty text or a case-insensitive whole-field placeholder (`-`, `nan`, `none`, `null`, `n/a`, `na`, `unknown`, `missing`, `timeout`, `timed out`, `error`, `failed`, `fail`, `fail process`, `failed process`, or `process failed`); apply Unicode NFKC; then case-fold. Source keys compare the exact ordered `(source_database, source_id)` pair, and MOFid keys compare the complete string—never a prefix or fuzzy match. MOFid equality is eligible only for `SUCCESS`, `SUCCESS_TOPOLOGY_UNKNOWN`, `SUCCESS_TOPOLOGY_ERROR`, or `SUCCESS_TOPOLOGY_TIMEOUT`. The release builder freshly recomputes every named release from all its current rows and imports no earlier component or MOFid edge. Missing, null, placeholder, unresolved-reconciliation, ambiguous-node, timeout, error, no-MOF, unmatched-node, and decomposition-error values add no edge and never match through a shared null. These operations never alter atoms, bonds, occupancies, coordinates, unit cells, chemical punctuation, or CIF bytes; the splitter consumes release group columns and does not repeat the canonicalization or relation construction.

**`identity_union`** selects the project-defined provisional source-ID/MOFid transitive groups. It reads `identity_status`, `identity_group`, and `identity_size`. For each named release, the release builder freshly recomputes direct edges over every current row from an exact canonicalized database-namespaced source ID or an eligible complete MOFid-v2/MOFid-v1 value, then takes connected-component closure; v26.0.2 does not import or seed an earlier v26.0.1 component. Each group, and therefore each value counted by `identity_size`, is one transitive connected component of structures joined by those identifier-equality edges, not a count of edges or identifiers. All three edge types have equal status and can bridge transitively; there is no precedence or conflict rule. Missing identifiers and non-success MOFid statuses add no edge and never match. It uses no RAC5, Zeo++, CrystalNets, CIF hash, common name, or StructureMatcher evidence; it is not proof of structural identity and is not an input to `priority_main` or `main_union`. A relation built from stage-only MOFid evidence remains a staged candidate and must be rebuilt if the authorized MOFid evidence changes.

Here **RAC5 fingerprint** means exact equality of all 264 finite ordered descriptor columns declared by the release after binary64 parsing, `-0.0` to `+0.0`, and `float.hex()` (`rtol=atol=0`), with no scaling, deletion, imputation, or rounding. The **selected Zeo++ fingerprint** applies the same numeric representation to density; LCD, PLD, and largest-free-path diameter; accessible/nonaccessible N₂ surface areas in m²/cm³ and m²/g; accessible/nonaccessible N₂ volumes in cm³/g and as fractions; and He void fraction, plus exact N₂ channel and available bonded-framework periodic dimensions. Its N₂/He probe radii are 1.655/1.32 Å; one invalid value/gate makes it unavailable; unit-cell-extensive values, OMS, topology, zero-probe values, and component counts are excluded.

The **current CrystalNets scientific fingerprint**, where current means topology evidence authorized by the loaded release rather than a search for newer runtime output, requires `SUCCESS`, `topology_available=true`, `error=null`, and complete nonempty SingleNodes/AllNodes subnets. It includes network dimension, interpenetrated-subnet count, catenation degree, subnet count, top-level single/all nets and agreement, and every canonically sorted subnet's agreement plus each node view's status, dimension, topology key/name, and genome. Count/catenation fields must equal subnet count and subnet indices must be unique/contiguous. Heterogeneous top-level summaries and node name/genome may be null and are retained. It hashes canonical sorted-key JSON, retaining duplicate subnets. Runtime, CIF paths/hashes, diagnostics, software boilerplate, and original subnet order are excluded; incomplete/failed results add no edge.

The optional method names and prefixes are project-defined labels, not scientific values. **`rac5_crystalnets`** reads `rac_crystalnets_status/group/size`; **`RT-`** identifies exact equality of all 264 finite RAC5 values plus that complete successful CrystalNets fingerprint. **`mofid_v2_crystalnets`** reads `mofid2_crystalnets_status/group/size`; **`M2T-`** identifies exact equality of the complete canonical eligible MOFid-v2 string plus that same fingerprint. Eligible MOFid-v2 status is exactly `SUCCESS`, `SUCCESS_TOPOLOGY_UNKNOWN`, `SUCCESS_TOPOLOGY_ERROR`, or `SUCCESS_TOPOLOGY_TIMEOUT`; the latter two are successful calculated identifiers whose embedded topology qualifier is ERROR or TIMEOUT, not MOFid execution failures. Every other MOFid-v2 status and every incomplete CrystalNets input adds no edge. M2T remains provisional whenever the release-authorized MOFid-v2 input is provisional. If the release-authorized MOFid-v2 values change, rebuild the M2T groups before use. **`SM-`** identifies a convenience component from direct strict edges. Prefix digests are criterion-bound length-delimited UTF-8 SHA-256, starting at eight uppercase hex characters and extending only on a collision. Strict SM uses Python 3.9, pymatgen 2024.2.8, NumPy 1.26.4; exhaustively tries pairs within equal parsed `ElementComparator` fractional-composition hashes; and requires forward/reverse `fit(..., symmetric=True)` under `ltol=stol=0.001`, `angle_tol=0.01`, `primitive_cell=true`, `scale=false`, `attempt_supercell=true`, `allow_subset=false`, `supercell_size=num_sites`, and no ignored species. Its pinned parser expands symmetry, uses site/frac tolerances 1e-4, checks occupancy, sorts, preserves disorder, and never repairs chemistry. Parser failures, timeouts, OOMs, matcher errors, and asymmetric results are `NOT_AVAILABLE` rather than unmatched and add no edge. Forward/reverse normalized RMS/max divide periodic displacement by that direction's `(V/Nsites)^(1/3)`, are dimensionless, and are neither angstrom RMSD nor `charnley/rmsd` output. Direct edges are authoritative and symmetric; an SM component is a convenience view and not an all-pairs or duplicate-identity claim. Missing, nonfinite, partial, timed-out, failed, or otherwise incomplete RAC5, MOFid, or CrystalNets input creates no `RT-`/`M2T-` evidence and unavailable rows never match. All three are optional references excluded from `priority_main` and `main_union`; the recommended explanatory precedence remains exact RAC5, then complete MOFid v2, then complete MOFid v1.

Parent methods serve different purposes:

- direct sensitivity: `rac5`, `mofid_v2`, `mofid_v1`;
- reference: `rac5_zeo`, `zeo`, `source_id`, `common_name`, `identity_union`, and `none`;
- optional non-decisive references, only when declared by the release: `rac5_crystalnets`, `mofid_v2_crystalnets`, and `structure_matcher_strict`; and
- computed: recommended explanatory `priority_main` and the separate broad leakage-only `main_union`, whose five exact edge types are defined in Section 7.

The CrystalNets-combined methods require an exact current CrystalNets fingerprint in addition to RAC5 or the full canonicalized MOFid-v2 value. Strict StructureMatcher uses audited direct symmetric pair edges; its connected components are a convenience view because tolerance matching can be non-transitive. Check clique/completeness diagnostics and the direct edge ledger before interpreting a component. The historical relaxed matcher is documentation-only and is never selectable. Optional methods do not alter `priority_main` or `main_union`.

In [ ]:
resolver = ParentResolver(dataset)
preview_ids = modelling_subset.structure_ids[:12]
parent_resolution = resolver.resolve(
    "priority_main", structure_ids=preview_ids
)

for structure_id in preview_ids:
    print(
        structure_id,
        parent_resolution.groups.get(structure_id),
        parent_resolution.evidence_by_id.get(structure_id),
        parent_resolution.exclusions.get(structure_id),
    )
print("Relevant conflict groups in preview:", len(parent_resolution.conflicts))

In [ ]:
declared_parent_methods = dict(
    dataset.parent_group_methods.get("csv_column_prefixes", {})
)
optional_reference_methods = (
    "rac5_crystalnets", "mofid_v2_crystalnets", "structure_matcher_strict"
)
available_optional_methods = tuple(
    method for method in optional_reference_methods if method in declared_parent_methods
)
print("Declared parent methods: ", tuple(declared_parent_methods))
print("Available optional refs:  ", available_optional_methods)
for method in available_optional_methods:
    optional_preview = resolver.resolve(method, structure_ids=preview_ids)
    print(method, "groups=", len(set(optional_preview.groups.values())), "excluded=", len(optional_preview.exclusions))

## 10. Build the recommended leakage-safe split

This section intentionally returns to the broader full `five_checker` view and applies only the COD/SI and CR/NCR filters; it does not reuse the Cu/Zn and ASR/FSR restrictions in `modelling_subset`. To split that exact smaller cohort instead, call `modelling_subset.train_valid_test_split(...)`; its preselection will be recorded in the receipt.

Here `leakage_guard="auto"` resolves to `main_union` for `priority_main`. The indivisible components are built on the full COD+CSD+SI universe from full CIF SHA-256, database-namespaced source siblings, and available release-authorized RAC5, MOFid v2, and MOFid v1 groups before experiment filters are applied. This recommended guard requires `manifests/cif_manifest.csv` with one full SHA-256 value for every release structure.

The allowed guards are `auto`, `parent_only`, and `main_union`. `auto` chooses `main_union` for `priority_main` and `parent_only` for an explicit direct/reference method. Use `parent_only` for a deliberate sensitivity experiment; request `main_union` explicitly when a reference-method experiment must still honor the broader production leakage graph. A source/label/target filter never reduces the universe used to build `main_union`.

In [ ]:
split = five_checker.train_valid_test_split(
    parent_method="priority_main",
    leakage_guard="auto",
    labels=("CR", "NCR"),
    sources=("COD", "SI"),
    fractions=(0.8, 0.1, 0.1),
    random_state=42,
    missing_parent="singleton",
    stratify_by=("label",),
)

print("Counts:            ", dict(split.counts))
print("Achieved fractions:", dict(split.achieved_fractions))
print("Labels by split:   ", {k: dict(v) for k, v in split.label_counts_by_split.items()})
print("Warnings:          ", split.warnings)

In [ ]:
audit = split.leakage_audit
assert audit["passed"]
assert audit["cross_split_block_count"] == 0
assert set(split.train_ids).isdisjoint(split.validation_ids)
assert set(split.train_ids).isdisjoint(split.test_ids)
assert set(split.validation_ids).isdisjoint(split.test_ids)

print("Resolved leakage guard:", split.leakage_guard)
print("Leakage blocks:       ", audit["block_count"])
print("Largest block:        ", audit["max_block_size"])
print("Crossed blocks:       ", audit["cross_split_block_count"])
print("Provisional input:    ", split.provisional_input)
print("Official split:       ", split.official_split)

## 11. Inspect and save the reproducibility receipt

Persist structure IDs rather than positional indices. The CSV contains every release row, including explicit exclusions. The JSON binds inputs, parameters, source-code hashes, assignments, warnings, parent conflicts, and the zero-leakage audit. The example uses `overwrite=True` so the tutorial is rerunnable; omit it in production when accidental replacement should fail closed.

In [ ]:
receipt = split.receipt()
assert receipt["parent_method_definition"]["project_defined_identifier"] is True
assert receipt["leakage_guard_definition"]["project_defined_identifier"] is True
assert receipt["requested_leakage_guard"] == "auto"
assert receipt["leakage_guard"] == "main_union"
receipt_summary = {
    "dataset_version": receipt["dataset_version"],
    "checker_view": receipt["checker_view"],
    "checker_view_kind": receipt["checker_view_kind"],
    "parent_method": receipt["parent_method"],
    "parent_method_definition": receipt["parent_method_definition"],
    "leakage_guard": receipt["leakage_guard"],
    "requested_leakage_guard": receipt["requested_leakage_guard"],
    "requested_leakage_guard_definition": receipt["requested_leakage_guard_definition"],
    "leakage_guard_definition": receipt["leakage_guard_definition"],
    "assignment_sha256": receipt["assignment_sha256"],
    "parent_conflict_count": receipt["parent_conflict_count"],
    "provisional_input": receipt["provisional_input"],
    "official_split": receipt["official_split"],
}
print(json.dumps(receipt_summary, indent=2))

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
csv_path, json_path = split.write(
    OUTPUT_ROOT,
    stem="cod_si_5checker_seed42",
    overwrite=True,
)

with csv_path.open(newline="", encoding="utf-8") as handle:
    assignment_rows = list(csv.DictReader(handle))
with json_path.open(encoding="utf-8") as handle:
    saved_receipt = json.load(handle)

assert len(assignment_rows) == len(dataset)
assert saved_receipt["assignment_sha256"] == receipt["assignment_sha256"]
assert saved_receipt["leakage_audit"]["passed"]

print("Assignment CSV:", csv_path)
print("Receipt JSON: ", json_path)
print("First assignment row:")
print(json.dumps(assignment_rows[0], indent=2))

## 12. Join split IDs to metadata and per-structure JSON

Use the stable IDs in `split.train_ids`, `split.validation_ids`, and `split.test_ids` to join targets with descriptors or CIFs. The per-structure JSON records provide the current checker detail and available RAC5, Zeo++, topology, and other release properties.

In [ ]:
train_preview = []
for structure_id in split.train_ids[:5]:
    item = dataset[structure_id]
    train_preview.append({
        "structure_id": structure_id,
        "target": split.labels[structure_id],
        "source_database": item.source_database,
        "structure_variant": item.structure_variant,
        "metal_elements": item.metal_elements,
        "cif_file": item.get("cif_file"),
    })
print(json.dumps(train_preview, indent=2))

example_id = split.train_ids[0]
structure_json_path = RELEASE_ROOT / "metadata" / "structures" / f"{example_id}.json"
if structure_json_path.is_file():
    with structure_json_path.open(encoding="utf-8") as handle:
        structure_json = json.load(handle)
    print("Per-structure JSON sections:", sorted(structure_json))
    print("Feature sections:           ", sorted(structure_json.get("features", {})))
else:
    print("This release does not include per-structure JSON files:", structure_json_path)

## 13. Optional determinism replay

The same inputs and parameters must reproduce the same `structure_id → partition` mapping. Enable this cell when auditing an environment or copied release.

In [ ]:
RUN_DETERMINISM_REPLAY = False

if RUN_DETERMINISM_REPLAY:
    repeated = five_checker.train_valid_test_split(
        parent_method="priority_main",
        leakage_guard="auto",
        labels=("CR", "NCR"),
        sources=("COD", "SI"),
        fractions=(0.8, 0.1, 0.1),
        random_state=42,
        missing_parent="singleton",
        stratify_by=("label",),
    )
    assert dict(repeated.assignments) == dict(split.assignments)
    assert repeated.receipt()["assignment_sha256"] == receipt["assignment_sha256"]
    print("Determinism replay passed.")
else:
    print("Skipped the optional second full split.")

## 14. Equivalent CLI command

The recommended split can also be generated without Python code:

```bash
coremof split /path/to/coremof_v26.0.2 \
  --checkers 5checker \
  --parent-method priority_main \
  --leakage-guard auto \
  --labels CR NCR \
  --sources COD SI \
  --fractions 0.8 0.1 0.1 \
  --random-state 42 \
  --output-directory model_splits \
  --stem cod_si_5checker_seed42
```

Use `--verify-cifs` for the expensive strict byte-level check. Do not pass `--official`: current releases do not yet provide an audited official assignment manifest, and the package intentionally fails closed.
The same target configuration used above can drive a merge and target-filtered split:

```bash
coremof merge-targets /path/to/coremof_v26.0.2 \
  --config /path/to/targets.json \
  --output-directory model_inputs

coremof split /path/to/coremof_v26.0.2 \
  --target-config /path/to/targets.json \
  --require-target co2_rosenbluth_weight \
  --require-target n2_rosenbluth_weight \
  --parent-method priority_main \
  --leakage-guard auto \
  --output-directory model_splits
```

## 15. Optional target-independent paired CR/NCR benchmark

This additive API leaves the historical split above unchanged. `main_union_plus_criteria` starts from the complete-release `main_union` graph (exact full CIF SHA-256, database-namespaced source sibling, and release-authorized exact RAC5/MOFid-v2/MOFid-v1 edges), adds co-membership edges from every ordered selected criterion, and takes connected-component closure before filtering. Missing criterion evidence is a structure-specific singleton. Each resulting effective leakage block is indivisible across partitions and is a leakage guard, not identity or common-parent proof.

`representative` is the versioned target-free diversity profile: complete 264-value finite RAC5 first; otherwise the 13 intensive N2/He Zeo++ fields plus N2 channel and bonded-framework dimensions; otherwise an explicit no-numeric tier. It applies median/interquartile-range scaling without imputation, retains zero-IQR fields with unit scale, reduces RAC5 to at most 32 principal components, then runs deterministic MiniBatchKMeans with sorted IDs, profile seed 2602, and `k=min(n,256,max(16,ceil(sqrt(n))))`. These strata balance source, variant, current topology category, and availability tier but never divide a leakage block. It requires exact NumPy 1.26.4, scikit-learn 1.5.0, SciPy 1.13.1, joblib 1.5.3, and threadpoolctl 3.6.0 from the `[benchmark]` extra; absence or version drift is an error, not a silent fallback. Numerical libraries run with a one-thread limit and their non-path runtime identity is receipted; cross-architecture bit identity is not guaranteed.

Strict five-checker CR means all five named checker results are available and PASS; strict five-checker NCR means all five are available and FAIL. `NOT_AVAILABLE` is a non-vote and never becomes NCR. The checksum-bound published v26.0.2 view has 6,294 raw CR and 2,299 raw NCR rows. With `group_criteria="priority_main"`, 1,601 CR and 572 NCR rows share a complete-release effective block with another label. The default fails closed; the explicit `complete_release_label_pure_effective_blocks` sensitivity policy leaves eligible C=4,693 and M=1,727. With NCR-pool fraction q, `full_cr` selects `round_half_up(q*M)` NCR and `C-round_half_up(q*M)` CR rows, always totaling C; q=1 uses all 1,727 eligible NCR plus 2,966 eligible CR rows. `fixed_pure_cr` reserves one common approximately 10% test from whole complete-release effective blocks containing only strict CR. It is identical across ratios/seeds and shares no effective block with train or validation. Increasing q within a seed adds an NCR prefix, removes a CR suffix, and never moves a persistent structure between partitions.

`RT` means exact complete finite 264-value RAC5 plus complete current-success CrystalNets equality; incomplete input adds no match. `M2T` replaces RAC5 with the complete canonical release-authorized MOFid-v2 string, where canonical text collapses Unicode whitespace, trims, rejects declared whole-field placeholders, applies Unicode NFKC, then case-folds without altering the CIF or chemistry; only `SUCCESS`, `SUCCESS_TOPOLOGY_UNKNOWN`, `SUCCESS_TOPOLOGY_ERROR`, and `SUCCESS_TOPOLOGY_TIMEOUT` are eligible. M2T remains provisional with provisional MOFid and must be rebuilt after authorized MOFid changes. `SM` is the convenience component over authoritative direct symmetric matches from the pinned strict pymatgen protocol; components are not all-pair duplicate claims, and failed/timeout/error inputs are unavailable rather than unmatched. RT, M2T, and SM are reference-only outside `priority_main` and `main_union` by themselves.

The supplementary `full_cr_diagnostic` covers the complete raw strict-CR pool, including rows excluded by the label-pure sensitivity policy, and reports exact-ID/same-effective-block overlap with training; it is not the independent paper test. Construction fails closed with exact counts if M>C or M>C-test_count. Every output remains exploratory with `official_split=false`. Targets attach only after assignment: `keep` preserves IDs and nulls, `error` requires completeness, and `drop` creates a filtered view without refill, rebalance, or resplit; target hashes never alter the frozen split receipt.

In [ ]:
RUN_CR_NCR_BENCHMARK = False

if RUN_CR_NCR_BENCHMARK:
    try:
        benchmark_suite = five_checker.build_cr_ncr_benchmark(
            ncr_pool_fractions=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            seeds=(42, 43, 44, 45, 46),
            total_size="full_cr",
            train=0.8,
            val=0.1,
            test=0.1,
            group_criteria="priority_main",
            cohort_eligibility="complete_release_label_pure_effective_blocks",
            diversity="representative",
            test_policy="fixed_pure_cr",
            include_full_cr_diagnostic=True,
        )
    except BenchmarkFeasibilityError as error:
        print("Benchmark stopped at the declared count gate:", error)
    else:
        print("Paired runs:", len(benchmark_suite.runs))
        print("Common clean-test count:", len(benchmark_suite.fixed_test_ids))
else:
    print("Skipped the optional dependency-complete CR/NCR benchmark.")

## Interpretation and redistribution reminders

- Treat `AMBIGUOUS` and `UNCHECKED` as distinct from NCR.
- Treat parent groups as criterion-dependent screening relations, not automatically as proof of identical frameworks.
- Keep the complete receipt with any published model or benchmark result.
- COD is the default open-data base. SI redistribution still requires asset-level rights review. CSD CIFs and structure-resolved CSD-derived data remain licence-gated unless CCDC grants permission for the intended redistribution.
- A source filter is not a licence-sanitization step: the standard assignment CSV deliberately contains all release rows, including excluded CSD rows. Prepare a separately audited public projection before redistribution.
- This handoff adds no restricted data to GitHub; the historical repository already contains legacy SI archives and is not a sanitized data-free clone. Do not add release tables, CIF archives, checker findings, structure-resolved targets, or generated assignment manifests. Transfer them through an approved manual or institutional channel after confirming recipient rights; verify the supplied SHA-256 ledgers.
- A future official v26.0.2 split must preserve frozen base assignments rather than independently reshuffling v26.0.1.

For the complete API, options, schemas, and troubleshooting guide, read the [dataset-splitting handbook](../README_DATASET_SPLITTING.md). For the separate-machine transfer sequence, read the [GPU benchmark handoff guide](../ML_BENCHMARK_HANDOFF.md).